In [ ]:
pip install ucimlrepo

In [118]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix


In [ ]:
# fetch dataset
student_performance = fetch_ucirepo(id=320)

# data (as pandas dataframes)
X = student_performance.data.features
y = student_performance.data.targets

# metadata
print(student_performance.metadata)

# variable information
print(student_performance.variables)

# 1. ESTATÍSTICAS DESCRITIVAS

In [ ]:
df = pd.concat([X, y], axis=1)
print("=== Estatísticas Descritivas ===")
display(df.describe(include='all'))

# 2. TRANSFORMAÇÕES DE LINHA E COLUNA

In [121]:
df = df[df['absences'] <= 30]
df['Target'] = (df['G3'] >= 6).astype(int)
df = df.drop(columns=['G1', 'G2', 'G3'])
df = pd.get_dummies(df, drop_first=True)

# 3. DIVISÃO EM TRÊS SUBCONJUNTOS

In [116]:
X = df.drop('Target', axis=1)
y = df['Target']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)


# 4. TREINAMENTO E AVALIAÇÃO DO MODELO

In [ ]:
modelo = RandomForestClassifier(random_state=42)
modelo.fit(X_train, y_train)

pred_val = modelo.predict(X_val)
print(f"\nAcurácia na Validação: {accuracy_score(y_val, pred_val):.2f}")

pred_test = modelo.predict(X_test)
print(f"Acurácia no Teste: {accuracy_score(y_test, pred_test):.2f}")

print("\n=== Matriz de Confusão ===")
print(confusion_matrix(y_test, pred_test))

# 5. PREDIÇÃO DO MODELO IMPLANTADO

In [ ]:
amostra = X_test.iloc[[0]]
resultado = "Aprovado" if modelo.predict(amostra)[0] == 1 else "Reprovado"
print(f"\n=== Resultado de Predição ===")
print(f"O aluno de teste foi classificado como: {resultado}")